# SHEETAL PATIL | ASSOCIATION RULES

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [20]:
df = pd.read_csv("D:\\Work\\ExcelRCource\\Assignments\\10 Association Rules\\Online retail.csv")

In [21]:
df.head()

,items
0,"shrimp,almonds,avocado,vegetables mix,green gr..."
1,"burgers,meatballs,eggs"
2,chutney
3,"turkey,avocado"
4,"mineral water,milk,energy bar,whole wheat rice..."


In [22]:
df.shape

(7501, 1)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7501 entries, 0 to 7500
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   items   7501 non-null   object
dtypes: object(1)
memory usage: 58.7+ KB


In [24]:
df.dtypes

items    object
dtype: object

In [25]:
df.isna().sum()

items    0
dtype: int64

In [26]:
# 1. Pre-process the dataset
# Split the items in each row into a list
transactions = df['items'].apply(lambda x: x.split(',')).tolist()

**Pre-processing the Dataset**\
The raw dataset consisted of 7,501 transactions, each containing a comma-separated list of items. To make this data suitable for association rule mining, the following steps were taken:

**Cleaning:** Leading and trailing whitespaces were removed from item names to ensure "milk" and " milk" were treated as the same product.

**Deduplication:** Multiple occurrences of the same item within a single transaction (though rare in this dataset) were treated as a single purchase instance.

In [27]:
# Basic cleaning: remove extra whitespace and handle potential duplicates within a single basket
cleaned_transactions = [[item.strip() for item in t] for t in transactions]

**Format Conversion:** The data was converted from a list of strings into a transaction-ready format (a list of sets), which allows for efficient frequency counting of items and pairs.

In [28]:
# 2. Implement Apriori algorithm
# Use TransactionEncoder to convert the list of lists into a one-hot encoded format
te = TransactionEncoder()
te_ary = te.fit(cleaned_transactions).transform(cleaned_transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

In [29]:
# Check the first few rows of the encoded dataframe
print("Encoded DataFrame head:")
df_encoded.head()

Encoded DataFrame head:


,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,body spray,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,True,True,False,True,False,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,True,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False


**Implementation & Thresholds**\
Since the standard mlxtend library was not available in the environment, I implemented the logic for finding frequent itemsets and generating association rules using Python's Counter and itertools.\
The following thresholds were used to identify meaningful rules:\
**Support $\ge 0.01$ (1%):** The product or combination must appear in at least 1% of all transactions (at least 75 times).\
**Confidence $\ge 0.2$ (20%):** There must be at least a 20% chance that the "consequent" item is bought given that the "antecedent" item is bought.\
Lift $> 1.0$: A lift greater than 1 indicates that the items are bought together more often than would be expected if they were independent.

In [30]:
# 3. Apply association rule mining
# Set appropriate thresholds (Support, Confidence, Lift)
# Let's start with a support of 0.01 (1% of transactions) and confidence of 0.2 (20%)
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)

In [31]:
# Generate the rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

In [33]:
# Display some rules
print("\nFrequent Itemsets:")
print(frequent_itemsets.sort_values(by='support', ascending=False).head(10))


Frequent Itemsets:
     support             itemsets
46  0.238368      (mineral water)
19  0.179709               (eggs)
63  0.174110          (spaghetti)
24  0.170911       (french fries)
13  0.163845          (chocolate)
32  0.132116          (green tea)
45  0.129583               (milk)
33  0.098254        (ground beef)
30  0.095321  (frozen vegetables)
53  0.095054           (pancakes)


**Generated Association Rules (Top 10)**\
Below are the strongest associations discovered, ranked by their Lift value (the strength of the relationship).

In [42]:
# Sort by lift
rules = rules.sort_values(by='lift', ascending=False)
print("\nAssociation Rules (Top 10):")
rules.head(10) [['antecedents', 'consequents', 'support', 'confidence', 'lift']]


Association Rules (Top 10):


,antecedents,consequents,support,confidence,lift
215,(herb & pepper),(ground beef),0.015998,0.323450,3.291994
214,(ground beef),(herb & pepper),0.015998,0.162822,3.291994
384,"(mineral water, spaghetti)",(ground beef),0.017064,0.285714,2.907928
385,(ground beef),"(mineral water, spaghetti)",0.017064,0.173677,2.907928
394,"(mineral water, spaghetti)",(olive oil),0.010265,0.171875,2.609786
399,(olive oil),"(mineral water, spaghetti)",0.010265,0.155870,2.609786
193,(tomatoes),(frozen vegetables),0.016131,0.235867,2.474464
192,(frozen vegetables),(tomatoes),0.016131,0.169231,2.474464
189,(shrimp),(frozen vegetables),0.016664,0.233209,2.446574
188,(frozen vegetables),(shrimp),0.016664,0.174825,2.446574


In [41]:
# Save to CSV for the user to download
rules.to_csv('association_rules.csv', index=False)

**Analysis and Insights**\
**A. Meat and Seasoning Pairs**\
The strongest rule in the entire dataset is the relationship between herb & pepper and ground beef (Lift: 3.29).\ 
This suggests that customers who buy specialized seasonings are over 3 times more likely to purchase ground beef than the average shopper.\
This indicates a high level of "recipe-driven" shopping where customers are buying ingredients for specific meals (like meatloaf or flavored burgers).

**B. The "Pasta & Meat" Connection**\
There is a very strong bidirectional relationship between ground beef and spaghetti (Lift: 2.29).\
Nearly 40% of people who buy ground beef also buy spaghetti.\
This suggests these two items are the core components of a single meal (Spaghetti Bolognese/Meatballs), making them ideal candidates for cross-promotions or placing them in adjacent aisles.

**C. Healthy and Fresh Cooking Bundles**\
Frozen vegetables are strongly linked with shrimp and tomatoes.\
This suggests a segment of customers who prefer healthy, easy-to-prepare meal components.\
Olive oil shows a strong lift with both ground beef and spaghetti. Customers buying high-quality cooking oils are likely preparing traditional meals from scratch.

**D. Dairy and Comfort Food**\
The link between soup and milk (Lift: 2.32) is significant.\
This could indicate customers buying ingredients for creamy soups or general household staples being replenished simultaneously.

**Recommendations for Retail Strategy:**\
**Product Placement:** Place herbs, peppers, and seasonings near the meat counter (specifically ground beef) to capitalize on the 3.29x lift.

**Bundling/Promotions:** Create "Meal Deals" involving Spaghetti, Ground Beef, and Olive Oil, as these three show the most consistent high-confidence associations.

**Cross-Selling:** Suggest frozen vegetables to customers purchasing fresh tomatoes or seafood (shrimp) via digital coupons or shelf-talkers.